# Parameter Golf — Train SOTA + Error Analysis

Trains the SOTA model to saturation (~5 hours) then runs a full error analysis.

**Prerequisites:** Run `download_data.ipynb` first if data isn't on Drive yet.

**What this produces:**
- `final_model.pt` — trained model checkpoint
- `experiments/error_analysis.json` — per-token error breakdown
- Both saved to Google Drive for persistence

## 1. Setup

In [ ]:
import shutil, os, glob, torch

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

# Clone repos
%cd /content
if not os.path.exists(OFFICIAL):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
if not os.path.exists(AUX):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git
else:
    %cd {AUX}
    !git pull

# Install deps
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

# Copy our files into the official repo
shutil.copy2(f'{AUX}/train_gpt_aux.py', f'{OFFICIAL}/train_gpt_aux.py')
shutil.copy2(f'{AUX}/train_gpt_sota.py', f'{OFFICIAL}/train_gpt_sota.py')
if os.path.exists(f'{OFFICIAL}/aux_losses'):
    shutil.rmtree(f'{OFFICIAL}/aux_losses')
shutil.copytree(f'{AUX}/aux_losses', f'{OFFICIAL}/aux_losses')
for src_file in glob.glob(f'{AUX}/scripts/*.py'):
    os.makedirs(f'{OFFICIAL}/scripts', exist_ok=True)
    shutil.copy2(src_file, f'{OFFICIAL}/scripts/{os.path.basename(src_file)}')

# Link or copy data from Drive (if downloaded there) or download fresh
DRIVE_DATA = f'{DRIVE_DIR}/data/datasets/fineweb10B_sp1024'
LOCAL_DATA = f'{OFFICIAL}/data/datasets/fineweb10B_sp1024'
LOCAL_TOK = f'{OFFICIAL}/data/tokenizers'

if os.path.exists(DRIVE_DATA) and glob.glob(f'{DRIVE_DATA}/fineweb_val_*.bin'):
    # Symlink from Drive
    os.makedirs(f'{OFFICIAL}/data/datasets', exist_ok=True)
    if os.path.exists(LOCAL_DATA) and not os.path.islink(LOCAL_DATA):
        shutil.rmtree(LOCAL_DATA)
    if not os.path.exists(LOCAL_DATA):
        os.symlink(DRIVE_DATA, LOCAL_DATA)
    # Tokenizer
    DRIVE_TOK = f'{DRIVE_DIR}/data/tokenizers'
    if os.path.exists(DRIVE_TOK):
        os.makedirs(f'{OFFICIAL}/data', exist_ok=True)
        if os.path.exists(LOCAL_TOK) and not os.path.islink(LOCAL_TOK):
            shutil.rmtree(LOCAL_TOK)
        if not os.path.exists(LOCAL_TOK):
            os.symlink(DRIVE_TOK, LOCAL_TOK)
    n_train = len(glob.glob(f'{LOCAL_DATA}/fineweb_train_*.bin'))
    n_val = len(glob.glob(f'{LOCAL_DATA}/fineweb_val_*.bin'))
    print(f'Data from Drive: {n_train} train shards, {n_val} val shards')
else:
    # Download fresh (1 shard minimum)
    %cd {OFFICIAL}
    !python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1
    print('Downloaded 1 training shard (run download_data.ipynb for more)')

# Setup logs on Drive
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
logs_dir = f'{OFFICIAL}/logs'
if os.path.exists(logs_dir) and not os.path.islink(logs_dir):
    shutil.rmtree(logs_dir)
if not os.path.exists(logs_dir):
    os.symlink(f'{DRIVE_DIR}/logs', logs_dir)

# Verify
%cd {OFFICIAL}
!python3 -c "import py_compile; py_compile.compile('train_gpt_aux.py', doraise=True); print('Script OK')"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nGPU: {gpu_name} ({vram_gb:.0f}GB)')
print('Ready.')

## 2. Train SOTA Model (~5 hours)

Trains the baseline SOTA to full saturation (~7100 steps equivalent).
Skip this cell if `final_model.pt` already exists.

In [ ]:
%%time
%cd /content/parameter-golf

# === CONFIGURATION ===
WALLCLOCK = 18000      # 5 hours (seconds)
BATCH_TOKENS = 786432  # Full SOTA batch size
SEED = 42
# =====================

import os
if os.path.exists('final_model.pt'):
    size_mb = os.path.getsize('final_model.pt') / 1e6
    print(f'Model already exists: final_model.pt ({size_mb:.1f} MB)')
    print('Delete it to retrain, or skip to error analysis.')
else:
    print(f'Training SOTA baseline for {WALLCLOCK}s ({WALLCLOCK/3600:.1f} hours)...')
    print(f'Batch: {BATCH_TOKENS} tokens | Seed: {SEED}')
    print('=' * 60)
    !USE_AUX_LOSSES=0 USE_COMPILE=0 \
     RUN_ID=sota_full_train SEED={SEED} \
     ITERATIONS=20000 TRAIN_BATCH_TOKENS={BATCH_TOKENS} \
     VAL_LOSS_EVERY=500 MAX_WALLCLOCK_SECONDS={WALLCLOCK} \
     python3 train_gpt_aux.py

In [ ]:
# Save model checkpoint to Drive
import shutil
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

for f in ['final_model.pt', 'final_model.int6.ptz']:
    if os.path.exists(f):
        dst = f'{DRIVE_DIR}/{f}'
        shutil.copy2(f, dst)
        print(f'Saved {f} ({os.path.getsize(f)/1e6:.1f} MB) → Drive')

## 3. Error Analysis

Analyzes the trained model's per-token predictions to find:
- What % of tokens are easy/medium/hard/impossible
- Which token types the model struggles with
- Where in the sequence prediction is hardest
- Top-K accuracy (how often is the correct answer close)
- Confidence calibration
- What focal loss would actually do to the gradient distribution

Takes ~5 min on A100.

In [ ]:
%cd /content/parameter-golf

# Verify model exists
import os
if not os.path.exists('final_model.pt'):
    # Try loading from Drive
    drive_model = f'{DRIVE_DIR}/final_model.pt'
    if os.path.exists(drive_model):
        import shutil
        shutil.copy2(drive_model, 'final_model.pt')
        print(f'Loaded model from Drive ({os.path.getsize("final_model.pt")/1e6:.1f} MB)')
    else:
        print('ERROR: No model found. Run training cell first.')
        raise FileNotFoundError('final_model.pt')

# Run error analysis
!python3 scripts/error_analysis.py \
    --model final_model.pt \
    --max-sequences 500 \
    --batch-size 8 \
    --output experiments/error_analysis.json

In [ ]:
# Save results to Drive and display
import shutil, json
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

if os.path.exists('experiments/error_analysis.json'):
    shutil.copy2('experiments/error_analysis.json', f'{DRIVE_DIR}/results/error_analysis.json')
    print(f'Saved to Drive: {DRIVE_DIR}/results/error_analysis.json')

    # Show key findings
    with open('experiments/error_analysis.json') as f:
        report = json.load(f)

    print('\n' + '=' * 60)
    print('KEY FINDINGS')
    print('=' * 60)

    o = report['overall']
    print(f"\nMean loss: {o['mean_loss']:.4f}")
    print(f"Top-1 accuracy: {o['top1_accuracy']:.1%}")
    print(f"Top-5 accuracy: {o['top5_accuracy']:.1%}")
    print(f"Top-10 accuracy: {o['top10_accuracy']:.1%}")

    d = report['difficulty_distribution']
    print(f"\nDifficulty:")
    print(f"  Easy  (loss < 1):  {d['easy_pct']:.1f}%")
    print(f"  Medium (1-4):      {d['medium_pct']:.1f}%")
    print(f"  Hard  (4-8):       {d['hard_pct']:.1f}%")
    print(f"  Impossible (>=8):  {d['impossible_pct']:.1f}%")

    imp = report['improvement_opportunity']
    print(f"\nImprovement opportunity:")
    print(f"  Correct (top-1):       {imp['correct_top1_pct']:.1f}%")
    print(f"  Wrong but in top-10:   {imp['wrong_but_in_top10_pct']:.1f}%  ← target these")
    print(f"  Wrong, not in top-10:  {imp['wrong_not_in_top10_pct']:.1f}%  ← probably hopeless")